# 🚀 Nemotron-3-Nano Fine-tuning with NeMo AutoModel (BIRD SQL)

This notebook walks you through how to perform **Low Rank Adapation** (LoRA), a form of **Parameter Efficient Fine Tuning** (PEFT) of the NVIDIA Nemotron-3-Nano-30B model on the **BIRD SQL** text-to-SQL dataset using **NeMo AutoModel**. LoRA trains only a small set of adapter weights instead of the full model, so fine-tuning is faster and uses less memory while still adapting the model to your task.

You will use **two containers**: one to run this notebook for data prep, fine-tuning, merging the adapter into the base model, and testing. and a second container to deploy the merged model via **NIM** so you can send inference requests from this notebook.

---

## 📋 What You're Working With

| | |
|:--:|:--|
| 🤖 **Model** | `NVIDIA-Nemotron-3-Nano-30B-A3B-BF16` |
| 🛠️ **Framework** | NeMo AutoModel (Hugging Face–native) |
| 📐 **Method** | LoRA (Parameter-Efficient Fine-Tuning) |
| 📊 **Dataset** | BIRD SQL with schema (`xu3kev/BIRD-SQL-data-train` via `bird_sql.DatasetBIRD`) |

---

## ✅ Prerequisites

**Hardware:** 2× GPUs (e.g. H200, H100), sufficient storage for model and checkpoints.  
**Software:** Linux, Docker, NVIDIA Container Toolkit. Run this notebook **inside** the NeMo AutoModel container; then register and select the **Python (AutoModel venv)** kernel (Step 1).

---

## 🗺️ Workflow at a Glance

| Step | What you'll do |
|:--:|:--|
| **1** | 🐳 Pull and run the NeMo AutoModel container (mount this repo); register and select **Python (AutoModel venv)** kernel for Jupyter |
| **2** | 📊 Prepare BIRD SQL dataset (train → JSONL) |
| **3** | ⚙️ Configure PEFT training (YAML) |
| **4** | 🎯 Run fine-tuning inside the container |
| **5** | 🔀 Merge LoRA adapter into base model → `merged_model/` for NIM |
| **6** | Deploy merged model using NIM |
| **7** | Inference on NIM: to test out our model |

---

## Step 1: Run with the NeMo AutoModel Container

We highly recommend that you run this tutorial inside the official [NeMo AutoModel container](https://catalog.ngc.nvidia.com/orgs/nvidia/containers/nemo-automodel). This image is specially configured to run NeMo AutoModel, PyTorch, and all of their dependencies for fine-tuning Nemotron-3-Nano.

### Launch the container (on your host)

Run these **on your host**. They mount your repo at `/workspace` and start the notebook container with **host networking** so Jupyter is reachable at `http://localhost:8888` without port forwarding. Set `REPO_PATH` to the path to your repo (the directory that contains use-case-examples).

```bash
export REPO_PATH=/path/to/root/directory
docker pull nvcr.io/nvidia/nemo-automodel:25.11.00
docker run -it --gpus all --network host --name notebook \
  -v "${REPO_PATH}:/workspace" nvcr.io/nvidia/nemo-automodel:25.11.00 bash
```

If the container is already running, attach with `docker exec -it notebook bash` instead of `docker run`. 


### Register the AutoModel venv kernel (inside the container)

**Do this first**, before starting Jupyter, so Jupyter sees the kernel when it starts and you don't need to restart it. Inside the container run:

```bash
source /opt/venv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name automodel-venv --display-name "Python (AutoModel venv)"
```


### Start Jupyter (inside the container)

Then start Jupyter:

```bash
cd /workspace/use-case-examples/sql-lora-finetuning-and-deployment
jupyter notebook --ip=0.0.0.0 --allow-root
```

From your machine, open **http://localhost:8888** and paste the token Jupyter printed. With host networking, the notebook is accessible without port forwarding. In the notebook, select **Kernel → Change kernel → Python (AutoModel venv)** so the notebook uses the correct environment.

In [ ]:
# Verify environment (run inside the container):
import sys
try:
    import nemo_automodel
    import datasets
    print("nemo_automodel:", nemo_automodel.__file__)
    print("datasets OK")
except ImportError as e:
    print("ImportError (run this notebook inside the container):", e)
try:
    import mamba_ssm
    print("mamba_ssm OK")
except ImportError as e:
    print("mamba_ssm:", e)


**If the cell above reports `No module named 'nemo_automodel'`:** Usually the notebook is using the wrong kernel; switch to **Kernel → Python (AutoModel venv)**.

### Accessing Jupyter from your browser

With **host networking**, Jupyter is on the host's port 8888. Get the token from the terminal where Jupyter is running, then open **http://localhost:8888/tree?token=YOUR_TOKEN** in your browser (replace `YOUR_TOKEN` with the token).

## Step 2: Prepare the BIRD SQL Dataset

We'll use the **BIRD SQL** dataset (`xu3kev/BIRD-SQL-data-train`). Each row is a text-to-SQL example with:

| Column | Description |
|--------|-------------|
| **`db_id`** | Database name (e.g. `video_games`) |
| **`question`** | Natural language question (e.g. "List down at least five publishers…") |
| **`evidence`** | Additional helpful context linking the question to the schema (e.g. "publishers refers to publisher_name…") |
| **`schema`** | DDL for the database (table/column definitions) |
| **`SQL`** | Target SQL query we want the model to generate |

For training we need a single **prompt** (input) and a single **target** (output). We don't use this raw format directly: we **map** it into one example per row with:

- **Input:** The full prompt the model sees: schema + question + evidence, formatted with the Nemotron chat template.
- **Output:** The assistant reply only: the SQL query plus an end-of-turn marker.

**`bird_sql.DatasetBIRD`** does that mapping: it loads BIRD from Hugging Face, applies the chat template, and writes one JSONL record per example with columns **`input`** and **`output`**. That JSONL is what the rest of the pipeline (and later, the YAML dataset config) expects. The cell below runs that preparation and saves **`dataset/training.jsonl`** for AutoModel.


In [ ]:
import os
import sys
from datasets import disable_caching

disable_caching()

# Ensure bird_sql is importable (cwd or repo-relative path)
cwd = os.getcwd()
for candidate in [cwd, os.path.join(cwd, "use-case-examples", "sql-lora-finetuning-and-deployment")]:
    bird_sql_dir = os.path.join(candidate, "bird_sql")
    if os.path.isdir(bird_sql_dir):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        break
from bird_sql.dataset_bird import DatasetBIRD

DATASET_DIR = os.environ.get("DATASET_DIR", os.path.join(os.getcwd(), "dataset"))
os.makedirs(DATASET_DIR, exist_ok=True)
training_jsonl = os.path.join(DATASET_DIR, "training.jsonl")

model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
max_seq_len = 4096
num_workers = 4

print("Preparing BIRD training dataset (with schema) via bird_sql.DatasetBIRD...")
dataset = DatasetBIRD(
    model_id_to_prep_for=model_id,
    max_seq_len=max_seq_len,
    num_workers=num_workers,
).make_dataset()
dataset = dataset.sort("length")
# Save only input/output for AutoModel JSONL (same shape as before)
dataset = dataset.select_columns(["input", "output"])
dataset.to_json(training_jsonl, orient="records", lines=True, force_ascii=False)
print(f"Saved {len(dataset)} examples to {training_jsonl}")

## Step 3: PEFT Training Config (YAML)

Create a YAML recipe that specifies the Nemotron-3-Nano model and your BIRD JSONL dataset. The recipe specifies a LoRA config in the `peft` section that will freeze the base model and only tune a parameter-efficient subset of the model as opposed to full-weights SFT.  The dataset is loaded via **ColumnMappedTextInstructionDataset** with `input` → question and `output` → answer, and **answer-only loss**. Runing the cell below will output the YAML file that we'll specify when running the fine-tuning script.

In [ ]:
import os

DATASET_DIR = os.environ.get("DATASET_DIR", os.path.join(os.getcwd(), "dataset"))
training_jsonl = os.path.join(DATASET_DIR, "training.jsonl")
config_path = os.path.join(os.getcwd(), "bird_peft_nemotron_nano.yaml")

yaml_content = f"""# PEFT (LoRA) fine-tuning: Nemotron-3-Nano on BIRD SQL (2 GPUs)
# Run from this directory inside the container: torchrun --nproc-per-node=2 finetune.py

step_scheduler:
  global_batch_size: 8
  local_batch_size: 4
  ckpt_every_steps: 200
  val_every_steps: 200
  max_steps: 1000
  num_epochs: 1

dist_env:
  backend: nccl
  timeout_minutes: 60

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  torch_dtype: bf16

checkpoint:
  enabled: true
  checkpoint_dir: checkpoints/
  model_save_format: safetensors
  save_consolidated: false

peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  match_all_linear: true
  dim: 8
  alpha: 32
  use_triton: true

distributed:
  _target_: nemo_automodel.components.distributed.fsdp2.FSDP2Manager
  dp_size: null
  tp_size: 1
  cp_size: 1
  sequence_parallel: false
  backend: nccl

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  # ColumnMappedTextInstructionDataset properly maps the dataset columns to what AutoModel expects
  path_or_dataset_id: dataset/training.jsonl
  # split is ignored for local JSONL; omit or set for HF datasets
  split: train
  column_mapping:
    question: input
    answer: output
  answer_only_loss_mask: true

packed_sequence:
  packed_sequence_size: 0

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: false

optimizer:
  _target_: torch.optim.Adam
  betas: [0.9, 0.999]
  eps: 1e-8
  lr: 1.0e-5
  weight_decay: 0
"""

with open(config_path, "w") as f:
    f.write(yaml_content)
print(f"Wrote config to {config_path}")
print(f"Dataset path in config: dataset/training.jsonl")

**Note:** If the dataset step fails on `split`, try removing or setting `split: null` in the YAML.

**Parameters you can change** (in the YAML above or in `bird_peft_nemotron_nano.yaml`):

- **`global_batch_size`** (default 8): Total examples per optimizer step across all GPUs. Larger values give stabler gradients but need more GPU memory. 8 fits on 2 GPUs for this model; reduce if OOM.
- **`local_batch_size`** (default 4): Examples per GPU per step. Must divide `global_batch_size`; typically `global_batch_size / num_gpus`. Set to 4 so each of 2 GPUs gets 4 samples.
- **`ckpt_every_steps`** (default 200): Save a checkpoint every N steps. 200 gives a few checkpoints in a 1000-step run without writing too often.
- **`max_steps`** (default 1000): Stop training after this many steps. 1000 keeps the run to about 1–2 hours; lower for a quicker test, higher for more training.
- **`peft.dim`** (default 8): LoRA rank (width of adapter matrices). **`peft.alpha`** (default 32): LoRA scaling (often 2× or 4× dim). dim=8, alpha=32 is a common balance of capacity vs overfitting and memory.
- **`optimizer.lr`** (default 1e-5): Learning rate. 1e-5 is a typical starting point for LoRA on LLMs; lower for stability, higher for faster (but riskier) convergence.


## Step 4: Run Fine-Tuning

Run the fine-tuning by invoking the `finetune.py` script. The number of training steps is set in the YAML (Step 3); expect the training to take **about 1–2 hours**. This model needs a minimum amount of GPU memory; if you hit an **out-of-memory** error or want a **shorter run**, edit the YAML in Step 3 (e.g. reduce `global_batch_size` / `local_batch_size` or lower `max_steps`). Adjust `--nproc-per-node` if you have a different GPU count.

```bash
cd /workspace/use-case-examples/sql-lora-finetuning-and-deployment
torchrun --nproc-per-node=2 finetune.py --config bird_peft_nemotron_nano.yaml
```
The script will need to be run inside the container. Note that fine tuning can also be initiated via the NeMo Automodel CLI:
```bash
automodel finetune llm -c bird_peft_nemotron_nano.yaml

## Step 5: Merge Adapter into Base Model for NIM

Next, we'll create a merged model by loading our trained adapter from the latest checkpoint in `checkpoints/`,  merging it with the base model via `peft`, and saving to `merged_model/` for Step 6. Expect this to take few minutes: it loads the full base model (~30B parameters) and the adapter, merges them in memory, then writes the fully merged model to disk.


In [ ]:
# Step 5 — Merge LoRA adapter into base model for NIM (Hugging Face format).
# Uses checkpoints/LATEST (points to the most recently saved checkpoint). 
import os
from pathlib import Path

RECIPE_DIR = Path("/workspace/use-case-examples/sql-lora-finetuning-and-deployment")
CHECKPOINT_PATH = (RECIPE_DIR / "checkpoints" / "LATEST").resolve() / "model"
MERGED_DIR = RECIPE_DIR / "merged_model"

assert CHECKPOINT_PATH.is_dir(), f"Checkpoint dir not found: {CHECKPOINT_PATH} (run Step 4 first and save a checkpoint)."

from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

print("Loading PEFT checkpoint (base model + adapter) from LATEST...")
model = AutoPeftModelForCausalLM.from_pretrained(str(CHECKPOINT_PATH), trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT_PATH))

print("Merging adapter into base model...")
model = model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
print(f"Saving merged model to {MERGED_DIR}...")
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Done. Use merged_model/ as the NIM workspace mount in Step 6.")

## Step 6: Set Up the NIM Container (Merged Model)

We deploy our model using a [NIM container](https://catalog.ngc.nvidia.com/orgs/nim/containers/nemotron-3-nano) so we can send inference requests from this notebook. With **host networking** (same as Step 1), NIM listens on the host's port 8000 and the notebook reaches it at `http://localhost:8000`. 

Copy-paste the block below into a **terminal on the host**. Set `REPO_PATH` to the host path you used in Step 1. You can obtain an NGC API key from [NGC Catalog](https://catalog.ngc.nvidia.com). Wait for NIM to finish launching before running Step 7.

**Note:** You may need to authenticate with NVCR via Docker before pulling the NIM image (e.g. if you see 401 Unauthorized):

```bash
echo $NGC_API_KEY | docker login nvcr.io -u '$oauthtoken' --password-stdin
```

**Deploy our fine-tuned merged model:**

```bash
export REPO_PATH=/path/to/root/directory
export NGC_API_KEY=your_ngc_key
export MERGED="${REPO_PATH}/use-case-examples/sql-lora-finetuning-and-deployment/merged_model"
docker run -it --rm --gpus all --network host --name nim \
  -e NGC_API_KEY=$NGC_API_KEY \
  -e NIM_DISABLE_MODEL_DOWNLOAD=1 \
  -e NIM_RELAX_MEM_CONSTRAINTS=1 \
  -e NIM_MODEL_PROFILE=7cbe1181600064c6e10ebaf843497acae35aacff2ab96fe8247ae541ae0ac28a \
  -v "${MERGED}:/opt/nim/workspace" \
  nvcr.io/nim/nvidia/nemotron-3-nano:1.7.0-variant
```



## Step 7: Inference on NIM

We send an HTTP request from this notebook to the NIM endpoint (`http://localhost:8000`) using one example from our training data as a smoke test. The cell prints the prompt, expected SQL, and the model's reply. NIM must be running (Step 6). Note that not all model generated SQL queries will be syntactically identical to the golden dataset.

In [ ]:
# Step 7 — Inference on NIM: one request using a training example.
import json
import os
import requests

NIM_BASE_URL = "http://localhost:8000"
NIM_MODEL_ID = "nvidia/nemotron-3-nano"
NIM_SYSTEM_FOR_SQL = (
    "detailed thinking off. You are a text-to-SQL assistant. "
    "Given a question and evidence about the database, respond with only the SQL query, "
    "no explanation, no markdown, no other text. "
    "Prefer the simplest query that answers the question: do not use table aliases (e.g. T1, T2) or JOINs unless the question truly requires multiple tables; when one table is enough, query it directly."
)
MAX_NEW_TOKENS = 512

INFERENCE_EXAMPLE_INDEX = 2
training_path = "dataset/training.jsonl" if os.path.exists("dataset/training.jsonl") else "training.jsonl"
if not os.path.exists(training_path):
    raise FileNotFoundError(f"Training file not found: {training_path}. Run Step 2 first.")
with open(training_path) as f:
    lines = [line for line in f if line.strip()]
example = json.loads(lines[INFERENCE_EXAMPLE_INDEX])
prompt_input = example["input"].strip()
expected = example["output"].strip()

def get_content_from_nim_response(data):
    try:
        choice = data.get("choices", [{}])[0]
        msg = choice.get("message", {})
        content = msg.get("content")
        if isinstance(content, str) and content:
            return content
        rc = msg.get("reasoning_content") or msg.get("reasoning")
        if isinstance(rc, str) and rc.strip():
            return rc.strip()
        return ""
    except Exception:
        return ""

url = f"{NIM_BASE_URL}/v1/chat/completions"
payload = {
    "model": NIM_MODEL_ID,
    "messages": [
        {"role": "system", "content": NIM_SYSTEM_FOR_SQL},
        {"role": "user", "content": prompt_input + "\n\nOutput only the SQL query, nothing else."},
    ],
    "max_tokens": MAX_NEW_TOKENS,
    "temperature": 0,
    "chat_template_kwargs": {"enable_thinking": False},
}

try:
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    content = get_content_from_nim_response(r.json())
    print("--- Input ---")
    print(repr(prompt_input))
    print("\n--- Expected (gold) ---")
    print(repr(expected))
    print("\n--- Model reply ---")
    print(repr(content if content else "(empty)"))
except requests.exceptions.ConnectionError as e:
    print("✗ Connection failed. Is NIM running? Start it (Step 6) with --network host")
    print("  Error:", e)
except Exception as e:
    print("Error:", e)

--- Input ---
'<|im_start|>system\n<|im_end|>\n<|im_start|>user\nCREATE TABLE IF NOT EXISTS "playstore"\n(\n    App              TEXT,\n    Category         TEXT,\n    Rating           REAL,\n    Reviews          INTEGER,\n    Size             TEXT,\n    Installs         TEXT,\n    Type             TEXT,\n    Price            TEXT,\n    "Content Rating" TEXT,\n    Genres           TEXT\n);\nCREATE TABLE IF NOT EXISTS "user_reviews"\n(\n    App                    TEXT\n        references "playstore" (App),\n    Translated_Review      TEXT,\n    Sentiment              TEXT,\n    Sentiment_Polarity     TEXT,\n    Sentiment_Subjectivity TEXT\n);\n\n\nHow many apps have rating of 5?\nFALSE;<|im_end|>\n<|im_start|>assistant\n<think></think>'

--- Expected (gold) ---
'SELECT COUNT(App) FROM playstore WHERE Rating = 5<|im_end|>'

--- Model reply ---
'SELECT SUM(CASE WHEN rating = 5 THEN 1 ELSE 0 END) FROM playstore'


---

## Conclusion

You now have a complete end-to-end workflow for fine-tuning and deploying NVIDIA Nemotron-3-Nano using NeMo AutoModel and the BIRD SQL dataset!

### What You've Accomplished

✅ **Fine-tuning:** Trained a custom text-to-SQL model using LoRA (PEFT) with the BIRD SQL dataset  
✅ **Merge & export:** Merged the LoRA adapter into the base model and saved to `merged_model/`  
✅ **Deployment:** Set up NIM (host network) so this notebook can call inference at `http://localhost:8000`

### Next Steps

Your fine-tuned model can now be:

- **Evaluated:** Run your model on the **BIRD SQL benchmark** (dev or test split) to measure text-to-SQL accuracy (e.g. execution accuracy or pass@k) and compare against the base model.
- **Used:** Send requests to the NIM endpoint from this notebook or any client on the same network
- **Improved:** Re-run Steps 2–4 with more data or different YAML settings (e.g. `max_steps`, `peft.dim`)
- **Shared:** Upload `merged_model/` to HuggingFace Hub for your team
- **Scaled:** Deploy NIM with more GPUs or replicate the NIM service for higher throughput

### API Endpoint

Once NIM is running (Step 6), the chat completions API is available at:

- **NIM (Nemotron-3-Nano):** `http://localhost:8000/v1/chat/completions`

Happy training and deploying! 🚀